# Gold Layer - Data Warehouse

## Objective

The Gold layer organizes the Silver dataset into a dimensional Data Warehouse using a Star Schema.

Instead of creating new features, this layer separates descriptive information into Dimension tables and transactional market metrics into a Fact table.

The resulting model is optimized for business intelligence dashboards, OLAP analysis, and reporting.

---

## Star Schema

### Dimensions

- Dim_Date
- Dim_Timeframe

### Fact

- Fact_Bitcoin_Market

---

## Input

- silver_bitcoin.csv

## Output

- dim_date.csv
- dim_timeframe.csv
- fact_bitcoin_market.csv

In [1]:
# Import libraries

import pandas as pd
import numpy as np

In [2]:
# Load Silver dataset

df = pd.read_csv(r"C:\Users\admin\Desktop\silver_bitcoin.csv", low_memory=False)
df["Open time"] = pd.to_datetime(df["Open time"], utc=True)
print(df.shape)


df.head()

(391385, 36)


,Open time,timeframe,year,quarter,month,day,Open,High,Low,Close,...,volume_spike,high_volatility,high_volatility_length,compression,overheating,whale_activity,breakout_up,breakout_down,support,resistance
0,2018-01-01 07:15:00+00:00,15m,2018,1,1,1,13774.99,13775.99,13600.00,13698.00,...,0,1,2,0,0,0,0,0,98.00,77.99
1,2018-01-01 07:30:00+00:00,15m,2018,1,1,1,13698.00,13775.00,13590.00,13644.95,...,0,1,3,0,0,0,0,0,54.95,130.05
2,2018-01-01 07:45:00+00:00,15m,2018,1,1,1,13644.97,13659.97,13555.02,13570.35,...,0,0,1,0,0,0,0,0,15.33,89.62
3,2018-01-01 08:00:00+00:00,15m,2018,1,1,1,13569.98,13665.00,13520.00,13656.23,...,0,1,1,0,0,0,0,0,136.23,8.77
4,2018-01-01 08:15:00+00:00,15m,2018,1,1,1,13656.23,13735.24,13610.27,13632.89,...,0,0,1,0,0,0,0,0,22.62,102.35


In [3]:
# Review dataset

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391385 entries, 0 to 391384
Data columns (total 36 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   Open time               391385 non-null  datetime64[ns, UTC]
 1   timeframe               391385 non-null  object             
 2   year                    391385 non-null  int64              
 3   quarter                 391385 non-null  int64              
 4   month                   391385 non-null  int64              
 5   day                     391385 non-null  int64              
 6   Open                    391385 non-null  float64            
 7   High                    391385 non-null  float64            
 8   Low                     391385 non-null  float64            
 9   Close                   391385 non-null  float64            
 10  price_change            391385 non-null  float64            
 11  return_pct              39

In [4]:
dim_date = (
    df[
        [
            "Open time",
            "year","timeframe",
            "quarter",
            "month",
            "day"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_date.insert(
    0,
    "date_key",
    range(1, len(dim_date)+1)
)

dim_date.head()

,date_key,Open time,year,timeframe,quarter,month,day
0,1,2018-01-01 07:15:00+00:00,2018,15m,1,1,1
1,2,2018-01-01 07:30:00+00:00,2018,15m,1,1,1
2,3,2018-01-01 07:45:00+00:00,2018,15m,1,1,1
3,4,2018-01-01 08:00:00+00:00,2018,15m,1,1,1
4,5,2018-01-01 08:15:00+00:00,2018,15m,1,1,1


In [5]:
dim_timeframe = (
    df[
        ["timeframe"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_timeframe.insert(
    0,
    "timeframe_key",
    range(1, len(dim_timeframe)+1)
)

dim_timeframe

,timeframe_key,timeframe
0,1,15m
1,2,1h
2,3,4h
3,4,1d


In [6]:
fact = df.merge(
    dim_date,
    on=[
        "Open time",
        "year","timeframe",
        "quarter",
        "month",
        "day"
    ]
)

fact = fact.merge(
    dim_timeframe,
    on="timeframe"
)

fact.head()

,Open time,timeframe,year,quarter,month,day,Open,High,Low,Close,...,high_volatility_length,compression,overheating,whale_activity,breakout_up,breakout_down,support,resistance,date_key,timeframe_key
0,2018-01-01 07:15:00+00:00,15m,2018,1,1,1,13774.99,13775.99,13600.00,13698.00,...,2,0,0,0,0,0,98.00,77.99,1,1
1,2018-01-01 07:30:00+00:00,15m,2018,1,1,1,13698.00,13775.00,13590.00,13644.95,...,3,0,0,0,0,0,54.95,130.05,2,1
2,2018-01-01 07:45:00+00:00,15m,2018,1,1,1,13644.97,13659.97,13555.02,13570.35,...,1,0,0,0,0,0,15.33,89.62,3,1
3,2018-01-01 08:00:00+00:00,15m,2018,1,1,1,13569.98,13665.00,13520.00,13656.23,...,1,0,0,0,0,0,136.23,8.77,4,1
4,2018-01-01 08:15:00+00:00,15m,2018,1,1,1,13656.23,13735.24,13610.27,13632.89,...,1,0,0,0,0,0,22.62,102.35,5,1


In [7]:
fact = fact[
[
"date_key",
"timeframe_key",

"Open",
"High",
"Low",
"Close",

"Volume",
"capital_flow",
"trade_activity",

"price_change",
"return_pct",
"volatility",

"buy_volume",
"sell_volume",
"buy_ratio",
"sell_ratio",
"order_flow",

"trend_direction",
"trend_length",
"trend_strength",

  "avg_volume_20",
"avg_volatility_20",


"volume_spike",
"high_volatility",
"high_volatility_length",
"compression",
"overheating",
"whale_activity",

"breakout_up",
"breakout_down",

"support",
"resistance"
]
]

fact.head()

,date_key,timeframe_key,Open,High,Low,Close,Volume,capital_flow,trade_activity,price_change,...,volume_spike,high_volatility,high_volatility_length,compression,overheating,whale_activity,breakout_up,breakout_down,support,resistance
0,1,1,13774.99,13775.99,13600.00,13698.00,74.453320,1.019043e+06,1312,-76.99,...,0,1,2,0,0,0,0,0,98.00,77.99
1,2,1,13698.00,13775.00,13590.00,13644.95,89.776654,1.228408e+06,895,-53.05,...,0,1,3,0,0,0,0,0,54.95,130.05
2,3,1,13644.97,13659.97,13555.02,13570.35,43.920484,5.972011e+05,814,-74.62,...,0,0,1,0,0,0,0,0,15.33,89.62
3,4,1,13569.98,13665.00,13520.00,13656.23,58.542067,7.948098e+05,919,86.25,...,0,1,1,0,0,0,0,0,136.23,8.77
4,5,1,13656.23,13735.24,13610.27,13632.89,58.900513,8.054309e+05,869,-23.34,...,0,0,1,0,0,0,0,0,22.62,102.35


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391385 entries, 0 to 391384
Data columns (total 36 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   Open time               391385 non-null  datetime64[ns, UTC]
 1   timeframe               391385 non-null  object             
 2   year                    391385 non-null  int64              
 3   quarter                 391385 non-null  int64              
 4   month                   391385 non-null  int64              
 5   day                     391385 non-null  int64              
 6   Open                    391385 non-null  float64            
 7   High                    391385 non-null  float64            
 8   Low                     391385 non-null  float64            
 9   Close                   391385 non-null  float64            
 10  price_change            391385 non-null  float64            
 11  return_pct              39

In [9]:
dim_date.to_csv(
    r"C:\Users\admin\Desktop\dim_date.csv",
    index=False
)

dim_timeframe.to_csv(
    r"C:\Users\admin\Desktop\dim_timeframe.csv",
    index=False
)

fact.to_csv(
    r"C:\Users\admin\Desktop\fact_bitcoin_market.csv",
    index=False
)